# UCF101 — Normal LSTM + Temporal Attention (Controlled Comparison)

This experiment tests a **unidirectional LSTM** using the original 51 pose features, with temporal attention. It is a clean comparison against the BiLSTM-attention model and does not overwrite any existing checkpoint.

- Uses the existing train/validation/test splits.
- Loads arrays with memory mapping to reduce RAM use.
- Selects the checkpoint using **validation macro-F1**.
- Evaluates window-level and video-level metrics at the end.
- Saves under new `normal_lstm_attention` filenames.

**Important:** do not use test metrics to tune this model. Compare video-level results with the previous BiLSTM video-level result (70.18% accuracy, 67.04% macro-F1).

In [1]:
from pathlib import Path
import json, random, time
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score

PROJECT_DIR = Path(r"C:\Users\admin\ViTPose_Action_Recognition")
DATA_DIR = PROJECT_DIR / "data" / "processed" / "sequences_all"
MODEL_DIR = PROJECT_DIR / "outputs" / "models"
REPORT_DIR = PROJECT_DIR / "outputs" / "reports"
MODEL_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
BATCH_SIZE = 64
EPOCHS = 50
PATIENCE = 8
LEARNING_RATE = 3e-4
WEIGHT_DECAY = 1e-4
HIDDEN_SIZE = 128
DROPOUT = 0.35
LABEL_SMOOTHING = 0.05
NUM_WORKERS = 0
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Device:", DEVICE)
print("Dataset folder:", DATA_DIR)
assert DATA_DIR.is_dir(), f"Dataset folder not found: {DATA_DIR}"


Device: cuda
Dataset folder: C:\Users\admin\ViTPose_Action_Recognition\data\processed\sequences_all


In [2]:
def load_array(name):
    path = DATA_DIR / name
    assert path.is_file(), f"Missing dataset file: {path}"
    return np.load(path, mmap_mode="r", allow_pickle=False)

X_train = load_array("X_train.npy")
y_train = np.asarray(load_array("y_train.npy"), dtype=np.int64)
X_val = load_array("X_val.npy")
y_val = np.asarray(load_array("y_val.npy"), dtype=np.int64)
X_test = load_array("X_test.npy")
y_test = np.asarray(load_array("y_test.npy"), dtype=np.int64)
videos_test = np.asarray(load_array("videos_test.npy"))

assert X_train.ndim == 3 and X_train.shape[2] == 51
assert X_train.shape[1:] == X_val.shape[1:] == X_test.shape[1:]
assert len(X_train) == len(y_train)
assert len(X_val) == len(y_val)
assert len(X_test) == len(y_test) == len(videos_test)

NUM_CLASSES = int(max(y_train.max(), y_val.max(), y_test.max()) + 1)
print("Train:", X_train.shape, "Val:", X_val.shape, "Test:", X_test.shape)
print("Classes:", NUM_CLASSES)


Train: (63533, 16, 51) Val: (20980, 16, 51) Test: (21009, 16, 51)
Classes: 101


## Data loaders

Use the existing pose features directly. No new motion features or augmentation are applied, so this experiment primarily tests the effect of using a unidirectional LSTM instead of a bidirectional one.

In [3]:
class PoseWindowDataset(Dataset):
    def __init__(self, X, y):
        self.X, self.y = X, y
    def __len__(self):
        return len(self.y)
    def __getitem__(self, idx):
        x = np.array(self.X[idx], dtype=np.float32, copy=True)
        return torch.from_numpy(x), torch.tensor(int(self.y[idx]), dtype=torch.long)

train_loader = DataLoader(
    PoseWindowDataset(X_train, y_train), batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=(DEVICE.type == "cuda")
)
val_loader = DataLoader(
    PoseWindowDataset(X_val, y_val), batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=(DEVICE.type == "cuda")
)
test_loader = DataLoader(
    PoseWindowDataset(X_test, y_test), batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=(DEVICE.type == "cuda")
)
print("Batches train/val/test:", len(train_loader), len(val_loader), len(test_loader))


Batches train/val/test: 993 328 329


## Define a unidirectional LSTM with temporal attention

Unlike a BiLSTM, this LSTM processes the sequence in one direction. Attention learns which time steps matter most for the final class prediction.

In [4]:
class NormalLSTMAttention(nn.Module):
    def __init__(self, input_size=51, hidden_size=128, num_classes=101, dropout=0.35):
        super().__init__()
        self.input_norm = nn.LayerNorm(input_size)
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=1,
            batch_first=True,
            bidirectional=False
        )
        self.attention = nn.Sequential(
            nn.Linear(hidden_size, hidden_size // 2),
            nn.Tanh(),
            nn.Linear(hidden_size // 2, 1)
        )
        self.classifier = nn.Sequential(
            nn.LayerNorm(hidden_size),
            nn.Dropout(dropout),
            nn.Linear(hidden_size, num_classes)
        )

    def forward(self, x):
        x = self.input_norm(x)
        sequence_output, _ = self.lstm(x)
        scores = self.attention(sequence_output).squeeze(-1)
        weights = torch.softmax(scores, dim=1)
        context = torch.sum(sequence_output * weights.unsqueeze(-1), dim=1)
        return self.classifier(context)

model = NormalLSTMAttention(
    input_size=51, hidden_size=HIDDEN_SIZE,
    num_classes=NUM_CLASSES, dropout=DROPOUT
).to(DEVICE)

criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="max", factor=0.5, patience=3
)
print(model)


NormalLSTMAttention(
  (input_norm): LayerNorm((51,), eps=1e-05, elementwise_affine=True)
  (lstm): LSTM(51, 128, batch_first=True)
  (attention): Sequential(
    (0): Linear(in_features=128, out_features=64, bias=True)
    (1): Tanh()
    (2): Linear(in_features=64, out_features=1, bias=True)
  )
  (classifier): Sequential(
    (0): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
    (1): Dropout(p=0.35, inplace=False)
    (2): Linear(in_features=128, out_features=101, bias=True)
  )
)


In [5]:
def run_epoch(loader, training):
    model.train(training)
    loss_total, true_list, pred_list = 0.0, [], []
    for xb, yb in loader:
        xb = xb.to(DEVICE, non_blocking=(DEVICE.type == "cuda"))
        yb = yb.to(DEVICE, non_blocking=(DEVICE.type == "cuda"))
        if training:
            optimizer.zero_grad(set_to_none=True)
        with torch.set_grad_enabled(training):
            logits = model(xb)
            loss = criterion(logits, yb)
            if training:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
        loss_total += loss.item() * len(yb)
        true_list.append(yb.detach().cpu().numpy())
        pred_list.append(logits.argmax(dim=1).detach().cpu().numpy())
    y_true = np.concatenate(true_list)
    y_pred = np.concatenate(pred_list)
    return {
        "loss": loss_total / len(loader.dataset),
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, labels=np.arange(NUM_CLASSES),
                             average="macro", zero_division=0)
    }


## Train with early stopping

Best checkpoint is selected using validation macro-F1. Training history is saved after every epoch.

In [6]:
BEST_PATH = MODEL_DIR / "normal_lstm_attention_best.pt"
LAST_PATH = MODEL_DIR / "normal_lstm_attention_last.pt"
HISTORY_PATH = REPORT_DIR / "normal_lstm_attention_history.csv"

history = []
best_val_f1 = -1.0
best_epoch = 0
stale_epochs = 0

for epoch in range(1, EPOCHS + 1):
    start = time.time()
    train_metrics = run_epoch(train_loader, training=True)
    val_metrics = run_epoch(val_loader, training=False)
    scheduler.step(val_metrics["macro_f1"])

    row = {
        "epoch": epoch,
        "train_loss": train_metrics["loss"],
        "train_accuracy": train_metrics["accuracy"],
        "train_macro_f1": train_metrics["macro_f1"],
        "val_loss": val_metrics["loss"],
        "val_accuracy": val_metrics["accuracy"],
        "val_macro_f1": val_metrics["macro_f1"],
        "learning_rate": optimizer.param_groups[0]["lr"],
        "seconds": time.time() - start
    }
    history.append(row)
    pd.DataFrame(history).to_csv(HISTORY_PATH, index=False)

    if val_metrics["macro_f1"] > best_val_f1:
        best_val_f1 = val_metrics["macro_f1"]
        best_epoch = epoch
        stale_epochs = 0
        torch.save({
            "model_state": model.state_dict(),
            "input_size": 51,
            "hidden_size": HIDDEN_SIZE,
            "num_layers": 1,
            "num_classes": NUM_CLASSES,
            "dropout": DROPOUT,
            "best_epoch": best_epoch,
            "best_val_macro_f1": best_val_f1,
            "seed": SEED
        }, BEST_PATH)
    else:
        stale_epochs += 1

    torch.save({
        "model_state": model.state_dict(),
        "epoch": epoch,
        "optimizer_state": optimizer.state_dict(),
        "best_epoch": best_epoch,
        "best_val_macro_f1": best_val_f1
    }, LAST_PATH)

    print(
        f"Epoch {epoch:02d}/{EPOCHS} | "
        f"train loss={train_metrics['loss']:.4f} acc={train_metrics['accuracy']:.4f} "
        f"F1={train_metrics['macro_f1']:.4f} | "
        f"val loss={val_metrics['loss']:.4f} acc={val_metrics['accuracy']:.4f} "
        f"F1={val_metrics['macro_f1']:.4f} | "
        f"best val F1={best_val_f1:.4f} epoch={best_epoch}"
    )

    if stale_epochs >= PATIENCE:
        print(f"Early stopping after {PATIENCE} epochs without validation macro-F1 improvement.")
        break

print("Best checkpoint:", BEST_PATH)
print("History:", HISTORY_PATH)


Epoch 01/50 | train loss=3.1315 acc=0.2945 F1=0.1660 | val loss=2.5708 acc=0.4237 F1=0.2567 | best val F1=0.2567 epoch=1
Epoch 02/50 | train loss=2.3867 acc=0.4665 F1=0.3106 | val loss=2.2714 acc=0.4963 F1=0.3407 | best val F1=0.3407 epoch=2
Epoch 03/50 | train loss=2.1001 acc=0.5416 F1=0.3936 | val loss=2.0932 acc=0.5426 F1=0.3938 | best val F1=0.3938 epoch=3
Epoch 04/50 | train loss=1.9170 acc=0.5893 F1=0.4479 | val loss=2.0044 acc=0.5669 F1=0.4309 | best val F1=0.4309 epoch=4
Epoch 05/50 | train loss=1.7956 acc=0.6216 F1=0.4857 | val loss=1.9309 acc=0.5863 F1=0.4588 | best val F1=0.4588 epoch=5
Epoch 06/50 | train loss=1.6872 acc=0.6517 F1=0.5238 | val loss=1.8934 acc=0.5950 F1=0.4689 | best val F1=0.4689 epoch=6
Epoch 07/50 | train loss=1.6123 acc=0.6740 F1=0.5523 | val loss=1.8539 acc=0.6070 F1=0.4871 | best val F1=0.4871 epoch=7
Epoch 08/50 | train loss=1.5507 acc=0.6940 F1=0.5749 | val loss=1.8238 acc=0.6163 F1=0.5006 | best val F1=0.5006 epoch=8
Epoch 09/50 | train loss=1.4951 

## Final test evaluation

Evaluates the selected checkpoint once. Video-level prediction is computed by averaging each source video's window probabilities.

In [7]:
checkpoint = torch.load(BEST_PATH, map_location=DEVICE, weights_only=False)
model.load_state_dict(checkpoint["model_state"])
model.eval()

prob_chunks, label_chunks = [], []
with torch.inference_mode():
    for xb, yb in test_loader:
        logits = model(xb.to(DEVICE, non_blocking=(DEVICE.type == "cuda")))
        prob_chunks.append(torch.softmax(logits, dim=1).cpu().numpy())
        label_chunks.append(yb.numpy())

window_probs = np.concatenate(prob_chunks)
window_labels = np.concatenate(label_chunks).astype(np.int64)
window_preds = window_probs.argmax(axis=1)

results = {
    "checkpoint": str(BEST_PATH),
    "best_epoch": int(checkpoint["best_epoch"]),
    "best_validation_macro_f1": float(checkpoint["best_val_macro_f1"]),
    "test_windows": int(len(window_labels)),
    "window_level_accuracy": float(accuracy_score(window_labels, window_preds)),
    "window_level_balanced_accuracy": float(balanced_accuracy_score(window_labels, window_preds)),
    "window_level_macro_f1": float(f1_score(
        window_labels, window_preds, labels=np.arange(NUM_CLASSES),
        average="macro", zero_division=0
    ))
}

unique_videos, inverse = np.unique(videos_test, return_inverse=True)
prob_sums = np.zeros((len(unique_videos), NUM_CLASSES), dtype=np.float64)
np.add.at(prob_sums, inverse, window_probs)
window_counts = np.bincount(inverse, minlength=len(unique_videos))
video_probs = prob_sums / window_counts[:, None]
video_labels = np.empty(len(unique_videos), dtype=np.int64)

for i in range(len(unique_videos)):
    labels_i = np.unique(window_labels[inverse == i])
    if len(labels_i) != 1:
        raise ValueError(f"Video {unique_videos[i]!r} has inconsistent labels: {labels_i}")
    video_labels[i] = labels_i[0]

video_preds = video_probs.argmax(axis=1)
results.update({
    "test_videos": int(len(unique_videos)),
    "video_level_accuracy": float(accuracy_score(video_labels, video_preds)),
    "video_level_balanced_accuracy": float(balanced_accuracy_score(video_labels, video_preds)),
    "video_level_macro_f1": float(f1_score(
        video_labels, video_preds, labels=np.arange(NUM_CLASSES),
        average="macro", zero_division=0
    ))
})

METRICS_PATH = REPORT_DIR / "normal_lstm_attention_test_metrics.json"
PREDICTIONS_PATH = REPORT_DIR / "normal_lstm_attention_video_predictions.csv"
with open(METRICS_PATH, "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2)
pd.DataFrame({
    "video_id": unique_videos.astype(str),
    "true_label": video_labels,
    "predicted_label": video_preds,
    "number_of_windows": window_counts,
    "correct": video_labels == video_preds
}).to_csv(PREDICTIONS_PATH, index=False)

print(json.dumps(results, indent=2))
print("Metrics saved:", METRICS_PATH)
print("Per-video predictions saved:", PREDICTIONS_PATH)


{
  "checkpoint": "C:\\Users\\admin\\ViTPose_Action_Recognition\\outputs\\models\\normal_lstm_attention_best.pt",
  "best_epoch": 39,
  "best_validation_macro_f1": 0.5693148422691843,
  "test_windows": 21009,
  "window_level_accuracy": 0.6396782331381788,
  "window_level_balanced_accuracy": 0.550724175369975,
  "window_level_macro_f1": 0.5538281892943282,
  "test_videos": 2441,
  "video_level_accuracy": 0.6558787382220401,
  "video_level_balanced_accuracy": 0.6258688258880373,
  "video_level_macro_f1": 0.6210531716570135
}
Metrics saved: C:\Users\admin\ViTPose_Action_Recognition\outputs\reports\normal_lstm_attention_test_metrics.json
Per-video predictions saved: C:\Users\admin\ViTPose_Action_Recognition\outputs\reports\normal_lstm_attention_video_predictions.csv


## Decision rule

- If video-level accuracy and macro-F1 are both below the BiLSTM baseline (70.18% and 67.04%), keep the BiLSTM.
- If the LSTM is slightly better on accuracy but worse on macro-F1, inspect class-wise performance before choosing it.
- If neither model reaches 75%, do not keep trying random hidden sizes. The next meaningful step is to inspect per-class errors and verify the split, then consider RGB+pose fusion if the project budget allows.
- The previous baseline's accuracy is video-level; compare it only with this notebook's `video_level_accuracy`, not its window-level score.